# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load and explore a dataset, defined by a Croissant schema, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset's Croissant schema is accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure the latest version of mlcroissant is installed
!pip install --quiet mlcroissant pandas

## 1. Data Loading

Load the dataset's metadata and content using `mlcroissant`. We'll also print the main metadata summary.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access the metadata (note: treat as object, not dict)
meta = dataset.metadata
print(f"\033[1m{meta.name}\033[0m\n")
print(meta.description)

# Display dataset's DOI and temporal & spatial coverage
print(f"\nDOI: {getattr(meta, 'identifier', None)}")
print(f"Spatial Coverage: {getattr(meta, 'spatialCoverage', None)}")
print(f"Temporal Coverage: {getattr(meta, 'temporalCoverage', None)}")

## 2. Data Overview

List all available record sets, fields, and their `@id` values. These entities form the core structure for data extraction in Croissant datasets.

_Note: The `@id` for each entity uniquely identifies it in the schema – **always refer to entities by their `@id`.**_

In [ ]:
# List all record sets in the dataset (by their @id and name)
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets detected in the top-level metadata. Attempting to list from underlying distributions...")
else:
    print(f"Found {len(record_sets)} record sets:")
    for rs in record_sets:
        print(f"  @id: {rs['@id']}  | name: {rs.get('name', 'N/A')}")

# If no record sets found in metadata, try auto-discovery from dataset.records()
if not record_sets:
    # Try iterating all possible record sets using dataset.records(record_set=None)
    print("Attempting to iterate through possible record sets (by inferring from records method)...")
    # mlcroissant can enumerate record sets using dataset.records(record_set=None)
    record_set_ids = []
    try:
        for rs_id in dataset.record_set_ids:
            print(f"  @id: {rs_id}")
            record_set_ids.append(rs_id)
    except Exception:
        print("Could not enumerate record sets via mlcroissant API.")
else:
    # If found, collect their @ids for later usage
    record_set_ids = [rs['@id'] for rs in record_sets]

# For demonstration, print fields for each record set:
print("\nSample fields for first available record set:")
if record_set_ids:
    first_rs = record_set_ids[0]
    # Get fields for the record set
    fields = dataset.fields(record_set=first_rs)
    if fields:
        for f in fields:
            print(f"  @id: {f['@id']}  | name: {f.get('name', 'N/A')}")
    else:
        print(f"  (no fields found for record set {first_rs})")

## 3. Data Extraction

Load tabular data from a specific `record_set` into a pandas DataFrame for analysis. Use only the `@id` of the desired record set (and, if needed, its fields by `@id`).

If there are multiple record sets, you may extract all or select specific ones by `@id`.

In [ ]:
# Extract data from each record set (@id)
dfs = {}
if record_set_ids:
    for rsid in record_set_ids:
        print(f"Extracting records from record set: {rsid}")
        records = list(dataset.records(record_set=rsid))
        if records:
            df = pd.DataFrame(records)
            dfs[rsid] = df
            print(f"  Columns: {df.columns.tolist()}")
        else:
            print(f"  No records found for record set {rsid}.")
    # Pick first record set for demo
    demo_rs = record_set_ids[0]
    print(f"\nSample rows from record set {demo_rs}:")
    display(dfs[demo_rs].head())
else:
    print("No available record sets to extract data from.")

## 4. Exploratory Data Analysis (EDA)

Perform common data processing: filter, normalize, and group records based on available fields in a selected record set. **All columns must be referenced by their `@id`.**

In [ ]:
# Pick the demo record set and use its first numeric field (by @id)
if dfs:
    df = dfs[demo_rs]
    print(f"Available columns (by @id) in demo record set: {df.columns.tolist()}")
    # Try to detect a likely numeric field
    numeric_field_id = None
    # Check datatypes
    for col in df.columns:
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
        except Exception:
            continue
    if numeric_field_id is not None:
        print(f"\nUsing numeric field (by @id): {numeric_field_id}")
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to find a group/categorical field
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() < len(df)//5:
                group_field = col
                break
        if group_field is not None:
            print(f"\nGrouping by field (by @id): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
            display(grouped_df.head())
        else:
            print("No suitable group (categorical) field detected.")
    else:
        print("No numeric field detected in selected record set for EDA.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization

Visualize the distribution of a chosen numeric field and, if possible, compare by groups. All columns must be referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dfs and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    # If grouping available
    if group_field is not None:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to load, inspect, and process a Croissant-formatted dataset with `mlcroissant`. We:

- Loaded metadata and explored key dataset information
- Listed available record sets and fields by their `@id`
- Loaded record data into DataFrames, referencing columns using their `@id`
- Applied filtering, normalization, and grouping based on field `@id`
- Generated basic exploratory visualizations

Refer to the dataset's Croissant schema and metadata for formal field definitions and further analysis opportunities.